# R Master v8a · Lap Head Graft Preview

基线：v7c body shell。目标：**Mona Master Body + Lapine Head/Face donor**。

这版只做安全预览，不做最终焊接：
- v7c 身体继续保留；
- 从用户自己的 Lapine unitypackage 提取 `Lapine.fbx`；
- 用 Head / Neck 骨作为对齐锚点，把 Lapine 头部组件对到 Mona；
- Mona 原头部在预览副本上删除，避免双头重叠；
- Lapine donor 保留 Face / Eye / Eyebrow / Hair / Mouth / Teeth / Tongue 等头部相关 mesh；
- 不改 Mona 505 骨；
- 不烘焙 Rest Pose；
- 不导 VRM；
- 不把 Lapine 原始资产写入 GitHub。

### Lapine 文件处理
Notebook 会先找：
`MyDrive/R_Master/reference/Lapine_Ver.1.11_A.zip`

如果已经缓存，以后全自动；如果第一次没有，只会弹出一次文件选择器。选中你自己的 `Lapine_Ver.1.11_A.zip` 后会自动缓存到 Drive，后续版本不再重复上传。

### 输出
- 全身正 / 侧 / 背 / 3/4
- 头肩正面 / 侧面 / 3/4
- 颈部接口近景
- report.json
- Review ZIP


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, tarfile, json, os, math

BUILD_TAG="v8a_lap_head_graft_20260919_r1"

print("R Master v8a · Lap Head Graft Preview")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v7c_refine"/"latest"/"R_Master_v7c_REFINED_PREVIEW.blend"
REF=ROOT/"reference"
CACHE=ROOT/"cache"
OUT=ROOT/"v8a_head_graft"/"latest"
REF.mkdir(parents=True,exist_ok=True); CACHE.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size<50*1024*1024:
    raise RuntimeError("没找到 v7c 预览文件。截图给二蛋即可。")
print(f"✓ v7c body：{SRC.stat().st_size/1024/1024:.1f} MiB")

LAPZIP=REF/"Lapine_Ver.1.11_A.zip"
if not LAPZIP.exists() or LAPZIP.stat().st_size<70*1024*1024:
    print("首次接 Lap 头：请选择你自己的 Lapine_Ver.1.11_A.zip。只需要这一次，之后缓存到 Drive。")
    up=files.upload()
    if not up: raise RuntimeError("没有选择 Lapine zip。")
    name=next(iter(up.keys()))
    tmp=Path("/content")/name
    tmp.write_bytes(up[name])
    shutil.copy2(tmp,LAPZIP)
    print("✓ Lapine 已缓存到 MyDrive/R_Master/reference/，以后不再重复上传")
else:
    print("✓ 复用 Drive 中的 Lapine 缓存")

import hashlib
h=hashlib.sha256()
with LAPZIP.open("rb") as f:
    for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
sha=h.hexdigest()
print("Lapine SHA256:",sha)
EXPECTED="8e215aeae8d2e42719305d66292b84c2714f616b7952591dee0383c376c86af4"
if sha!=EXPECTED:
    raise RuntimeError("Lapine zip SHA256 与已验证源不一致，停止。")
print("✓ Lapine 源校验通过")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v8a"); LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")

# 从 unitypackage 中只提取 Lapine.fbx，不公开、不写 GitHub。
UNPACK=LOCAL/"lapine_unpack"
if UNPACK.exists(): shutil.rmtree(UNPACK)
UNPACK.mkdir(parents=True)

outer=UNPACK/"outer"
with zipfile.ZipFile(LAPZIP,"r") as z:
    unity=[n for n in z.namelist() if n.lower().endswith("lapine.unitypackage")]
    if not unity: raise RuntimeError("zip 中没找到 Lapine.unitypackage")
    z.extract(unity[0],outer)
UNITY=outer/unity[0]

pkg=UNPACK/"pkg"; pkg.mkdir()
with tarfile.open(UNITY,"r:*") as t: t.extractall(pkg)

FBX=None
for p in pkg.glob("*/pathname"):
    try: path=p.read_text(encoding="utf-8").strip()
    except: continue
    if path=="Assets/Models/FBX/Lapine.fbx":
        candidate=p.parent/"asset"
        if candidate.exists():
            FBX=UNPACK/"Lapine.fbx"; shutil.copy2(candidate,FBX); break
if not FBX: raise RuntimeError("unitypackage 中没提取到 Assets/Models/FBX/Lapine.fbx")
print(f"✓ Lapine.fbx 临时提取：{FBX.stat().st_size/1024/1024:.1f} MiB")



In [ ]:
BUILD=LOCAL/"R_Master_v8a_Build.py"; RENDER=LOCAL/"R_Master_v8a_Render.py"
BUILD.write_text("\nimport bpy, os, sys, json, math\nfrom mathutils import Vector, Matrix\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=fbx=tag=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--fbx\" and i+1<len(argv): fbx=argv[i+1]\n    if a==\"--tag\" and i+1<len(argv): tag=argv[i+1]\nif not out or not fbx: raise RuntimeError(\"missing args\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or body.type!=\"MESH\": raise RuntimeError(\"Mona body missing\")\nif not rig or rig.type!=\"ARMATURE\": raise RuntimeError(\"Mona rig missing\")\n\n# Snapshot existing objects, then import donor.\nbefore=set(bpy.data.objects)\nbpy.ops.import_scene.fbx(filepath=fbx,automatic_bone_orientation=False)\ndonor_objs=[o for o in bpy.data.objects if o not in before]\ndonor_rigs=[o for o in donor_objs if o.type==\"ARMATURE\"]\nif not donor_rigs: raise RuntimeError(\"Lapine armature not found after FBX import\")\nlaprig=max(donor_rigs,key=lambda o:len(o.data.bones))\n\ndef bone_any(r,names):\n    for n in names:\n        b=r.pose.bones.get(n)\n        if b:return b\n    # suffix fallback\n    low=[x.lower() for x in names]\n    for b in r.pose.bones:\n        if any(b.name.lower().endswith(x.lower()) for x in names): return b\n    return None\n\nmh=bone_any(rig,[\"Head\",\"head\"])\nmn=bone_any(rig,[\"Neck2\",\"Neck1\",\"Neck\",\"neck\"])\nlh=bone_any(laprig,[\"Head\",\"head\"])\nln=bone_any(laprig,[\"Neck\",\"neck\"])\nif not all([mh,mn,lh,ln]): raise RuntimeError(\"Head/Neck anchor bones missing\")\n\nm_head=rig.matrix_world@mh.head\nm_neck=rig.matrix_world@mn.head\nl_head=laprig.matrix_world@lh.head\nl_neck=laprig.matrix_world@ln.head\nmvec=m_head-m_neck; lvec=l_head-l_neck\nscale=mvec.length/max(1e-6,lvec.length)\nrot=lvec.normalized().rotation_difference(mvec.normalized())\nR=rot.to_matrix().to_4x4()\nS=Matrix.Scale(scale,4)\nT1=Matrix.Translation(-l_neck)\nT2=Matrix.Translation(m_neck)\nX=T2@R@S@T1\n\n# Determine donor head meshes using names and spatial relation.\nmesh_donors=[o for o in donor_objs if o.type==\"MESH\"]\nhead_tokens=(\"face\",\"eye\",\"brow\",\"hair\",\"lash\",\"mouth\",\"teeth\",\"tooth\",\"tongue\",\"jaw\",\"ear\")\nselected=[]\nfor o in mesh_donors:\n    n=o.name.lower()\n    if any(t in n for t in head_tokens):\n        selected.append(o)\n\n# If explicit names are sparse, include meshes whose bbox is predominantly above Lapine neck.\nif len(selected)<3:\n    for o in mesh_donors:\n        if o in selected: continue\n        pts=[o.matrix_world@Vector(c) for c in o.bound_box]\n        if pts and sum(p.z>l_neck.z for p in pts)>=6:\n            selected.append(o)\n\nif not selected: raise RuntimeError(\"No Lapine head meshes selected\")\n\n# Apply alignment in world space, then detach donor armature modifiers for static preview.\nfor o in selected:\n    o.matrix_world=X@o.matrix_world\n    for mod in list(o.modifiers):\n        if mod.type==\"ARMATURE\" and mod.object==laprig:\n            o.modifiers.remove(mod)\n    o[\"R_DONOR\"]=\"LapineHead_v8a\"\n\n# Delete nonselected Lapine objects, including donor body/rig.\nfor o in list(donor_objs):\n    if o not in selected and o.name in bpy.data.objects:\n        bpy.data.objects.remove(o,do_unlink=True)\nif laprig.name in bpy.data.objects:\n    bpy.data.objects.remove(laprig,do_unlink=True)\n\n# Remove Mona head vertices in preview copy, leaving neck/body intact.\n# Uses Head bone weights + vertical guard above Mona neck.\nhead_groups=[]\nfor name in [\"Head\",\"HeadStretch\"]:\n    if name in body.vertex_groups: head_groups.append(body.vertex_groups[name].index)\n\nmw=body.matrix_world.copy()\nto_delete=[]\nfor v in body.data.vertices:\n    p=mw@v.co\n    d={g.group:g.weight for g in v.groups}\n    hw=max([d.get(i,0.0) for i in head_groups] or [0.0])\n    if p.z>m_neck.z+0.025 and hw>0.08:\n        to_delete.append(v.index)\n\nif not to_delete:\n    raise RuntimeError(\"Mona head removal selected 0 vertices\")\n\nbpy.context.view_layer.objects.active=body\nbody.select_set(True)\nbpy.ops.object.mode_set(mode=\"EDIT\")\nbpy.ops.mesh.select_all(action=\"DESELECT\")\nbpy.ops.object.mode_set(mode=\"OBJECT\")\nfor i in to_delete: body.data.vertices[i].select=True\nbpy.ops.object.mode_set(mode=\"EDIT\")\nbpy.ops.mesh.delete(type=\"VERT\")\nbpy.ops.object.mode_set(mode=\"OBJECT\")\nbody.select_set(False)\n\n# Keep only Mona body + selected donor meshes.\nfor o in list(bpy.data.objects):\n    if o.type==\"MESH\" and o!=body and o not in selected:\n        bpy.data.objects.remove(o,do_unlink=True)\n\nreport={\n \"ok\":True,\n \"stage\":\"R_Master_v8a_LapHeadGraftPreview\",\n \"build_tag\":tag,\n \"mona_body\":body.name,\n \"mona_rig\":rig.name,\n \"mona_bone_count\":len(rig.data.bones),\n \"lapine_selected_meshes\":[o.name for o in selected],\n \"lapine_selected_mesh_count\":len(selected),\n \"mona_head_vertices_removed\":len(to_delete),\n \"alignment\":{\n   \"uniform_scale\":float(scale),\n   \"mona_head_world\":list(map(float,m_head)),\n   \"mona_neck_world\":list(map(float,m_neck)),\n   \"lapine_head_world_before\":list(map(float,l_head)),\n   \"lapine_neck_world_before\":list(map(float,l_neck))\n },\n \"preview_only\":True,\n \"weights_transferred\":False,\n \"neck_welded\":False,\n \"rest_pose_baked\":False,\n \"final_vrm\":False\n}\nwith open(os.path.join(out,\"R_Master_v8a_report.json\"),\"w\",encoding=\"utf-8\") as f:\n    json.dump(report,f,ensure_ascii=False,indent=2)\n\nbpy.ops.wm.save_as_mainfile(filepath=os.path.join(out,\"R_Master_v8a_HEAD_GRAFT_PREVIEW.blend\"),check_existing=False)\nprint(\"[R Master v8a] BUILD_OK\")\nprint(\"[R Master v8a] donor meshes:\",report[\"lapine_selected_meshes\"])\nprint(\"[R Master v8a] removed Mona head verts:\",len(to_delete))\nprint(\"[R Master v8a] scale:\",scale)\n",encoding="utf-8")
RENDER.write_text("\nimport bpy, os, sys\nfrom mathutils import Vector\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--view\" and i+1<len(argv): view=argv[i+1]\nif not out or not view: raise RuntimeError(\"missing args\")\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a\"]\nif not body or not donors: raise RuntimeError(\"preview objects missing\")\nfor o in bpy.context.scene.objects:\n    if o.type==\"ARMATURE\": o.hide_render=True\n    if o.type==\"MESH\": o.hide_render=(o!=body and o not in donors)\npts=[]\nfor o in [body]+donors:\n    pts += [o.matrix_world@Vector(c) for c in o.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\ncenter=(mn+mx)*.5; height=mx.z-mn.z; dist=max(height,mx.x-mn.x,mx.y-mn.y)*2.6\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"; scene.render.image_settings.file_format=\"PNG\"\nscene.display.shading.light=\"STUDIO\"; scene.display.shading.show_shadows=True; scene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"; scene.display.shading.color_type=\"SINGLE\"; scene.display.shading.single_color=(.60,.60,.63)\nscene.display.shading.background_type=\"VIEWPORT\"; scene.display.shading.background_color=(.04,.04,.05)\ncd=bpy.data.cameras.get(\"R_v8a_Cam_DATA\") or bpy.data.cameras.new(\"R_v8a_Cam_DATA\")\ncam=bpy.data.objects.get(\"R_v8a_Cam\")\nif not cam:\n    cam=bpy.data.objects.new(\"R_v8a_Cam\",cd); scene.collection.objects.link(cam)\nscene.camera=cam; cam.data.type=\"ORTHO\"\ndef look(t): cam.rotation_euler=(Vector(t)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\ndef rr(fname,pos,target,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res; scene.render.resolution_percentage=100\n    cam.location=Vector(pos); cam.data.ortho_scale=scale; look(target); scene.render.filepath=os.path.join(out,fname)\n    bpy.ops.render.render(write_still=True)\nspec={\n \"front\":(\"R_Master_v8a_front.png\",(center.x,center.y-dist,center.z),center,height*1.08,(720,960)),\n \"side\":(\"R_Master_v8a_side.png\",(center.x+dist,center.y,center.z),center,height*1.08,(720,960)),\n \"back\":(\"R_Master_v8a_back.png\",(center.x,center.y+dist,center.z),center,height*1.08,(720,960)),\n \"three_quarter\":(\"R_Master_v8a_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08,(720,960)),\n}\nheadz=mx.z-height*.10\nspec[\"head_front\"]=(\"R_Master_v8a_head_front.png\",(center.x,center.y-dist,headz),(center.x,center.y,headz),height*.30,(900,900))\nspec[\"head_side\"]=(\"R_Master_v8a_head_side.png\",(center.x+dist,center.y,headz),(center.x,center.y,headz),height*.30,(900,900))\nspec[\"head_three_quarter\"]=(\"R_Master_v8a_head_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,headz),(center.x,center.y,headz),height*.30,(900,900))\nneckz=mx.z-height*.20\nspec[\"neck_interface\"]=(\"R_Master_v8a_neck_interface.png\",(center.x+dist*.70,center.y-dist*.70,neckz),(center.x,center.y,neckz),height*.24,(900,700))\nif view not in spec: raise RuntimeError(\"unknown view\")\nrr(*spec[view])\nprint(\"[R Master v8a] RENDER_OK\",view)\n",encoding="utf-8")
STAGE=OUT/"R_Master_v8a_HEAD_GRAFT_PREVIEW.blend"; REPORT=OUT/"R_Master_v8a_report.json"
rebuild=True
if STAGE.exists() and STAGE.stat().st_size>50*1024*1024 and REPORT.exists():
    try: rebuild=json.loads(REPORT.read_text(encoding="utf-8")).get("build_tag")!=BUILD_TAG
    except: rebuild=True
if rebuild:
    TMP=LOCAL/"stage"
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir(parents=True)
    logp=TMP/"R_Master_v8a_build.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(BUILD),"--","--out",str(TMP),"--fbx",str(FBX),"--tag",BUILD_TAG]
    with logp.open("w",encoding="utf-8") as log:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "R Master v8a" in line or "Traceback" in line or "Error" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(logp.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError(f"v8a 构建失败，退出码 {rc}")
    for n in ["R_Master_v8a_HEAD_GRAFT_PREVIEW.blend","R_Master_v8a_report.json","R_Master_v8a_build.log"]:
        shutil.copy2(TMP/n,OUT/n)
else:
    print("✓ v8a 构建已存在，跳过")
print(json.loads(REPORT.read_text(encoding="utf-8")))



In [ ]:
views=[
 ("front","R_Master_v8a_front.png"),("side","R_Master_v8a_side.png"),
 ("back","R_Master_v8a_back.png"),("three_quarter","R_Master_v8a_three_quarter.png"),
 ("head_front","R_Master_v8a_head_front.png"),("head_side","R_Master_v8a_head_side.png"),
 ("head_three_quarter","R_Master_v8a_head_three_quarter.png"),("neck_interface","R_Master_v8a_neck_interface.png")]
for i,(v,f) in enumerate(views,1):
    p=OUT/f
    if p.exists() and p.stat().st_size>20000:
        print(f"✓ [{i}/8] {v} 已存在"); continue
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER),"--","--out",str(OUT),"--view",v]
    r=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    imp=[x for x in r.stdout.splitlines() if "R Master v8a" in x or "Traceback" in x or "Error" in x]
    if imp: print("\n".join(imp[-10:]))
    if r.returncode!=0: raise RuntimeError(v+" 渲染失败")
print("✓ v8a 8 视图完成")



In [ ]:
from IPython.display import display,Image,Markdown
items=[
 ("全身正面","R_Master_v8a_front.png"),("全身侧面","R_Master_v8a_side.png"),
 ("全身背面","R_Master_v8a_back.png"),("全身 3/4","R_Master_v8a_three_quarter.png"),
 ("头肩正面","R_Master_v8a_head_front.png"),("头肩侧面","R_Master_v8a_head_side.png"),
 ("头肩 3/4","R_Master_v8a_head_three_quarter.png"),("颈部接口","R_Master_v8a_neck_interface.png")]
for title,f in items:
    display(Markdown("### "+title)); display(Image(filename=str(OUT/f),width=500))
zpath=OUT/"R_Master_v8a_Review.zip"
if zpath.exists(): zpath.unlink()
with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for _,f in items: z.write(OUT/f,arcname=f)
    for f in ["R_Master_v8a_report.json","R_Master_v8a_build.log"]: z.write(OUT/f,arcname=f)
print(f"✓ Review ZIP：{zpath.stat().st_size/1024/1024:.1f} MiB")
print("重 .blend 与 Lapine 源都留在你自己的 Drive，不下载到手机。")
files.download(str(zpath))

